# FIT5202 Data processing for Big data

## Week 4 Lab: Parallel Joins — Strategy, Data Movement and Adaptive Execution

Last week you followed a single DataFrame through Spark's execution machinery, this week we add a second DataFrame, and ask:

> **What does Spark have to do differently when a query needs matching rows from *two* DataFrames, rather than searching *one*?**

We will examine how Spark selects a physical join strategy using available statistics, and how Adaptive Query Execution can use runtime evidence to revise supported parts of a plan when necessary.


## Today's Plan

By the end of today's lab, you will be able to:
1. predict the result of an inner, left and full outer join, and identify repeated and unmatched rows;
2. distinguish a **logical join** (which rows must be returned) from a **physical join strategy** (how Spark actually produces them);
3. recognise `SortMergeJoin`, `BroadcastHashJoin`, `Exchange` and `BroadcastExchange` in a formatted physical plan;
4. explain why Spark chose a particular physical strategy for a given pair of DataFrames;
5. follow a join through the Spark UI's SQL/DataFrame, Stages and Tasks views;
6. compare a broadcast hash join and a sort-merge join, including their physical operators and data-movement requirements;
7. explain why changing the physical execution strategy does not change the logical join requested by the query;
8. explain conceptually how a hot join key can affect shuffle-based and broadcast joins differently;
9. audit a plausible-sounding optimisation suggestion using plan evidence, rather than accepting or rejecting it on reputation alone.


Let's get started.

## Table of Contents

1. [From Parallel Search to Parallel Join](#part-1)
2. [Inner and Outer Joins](#part-2)
3. [Logical Join vs Physical Strategy](#part-3)
4. [Spark Join Planning and AQE](#part-4)
5. [Follow the Join in the Spark UI](#part-5)
6. [Compare Broadcast Hash and Sort-Merge Joins](#part-6)
7. [Check an Optimisation Suggestion](#part-7)
8. [Take-Home Practice](#take-home)
9. [Stop Spark](#stop-spark)

<a id="part-1"></a>
# Part 1 — From Parallel Search to Parallel Join (10 mins)

Let's start Spark the same way we did in Weeks 2 and 3.

In [ ]:
from pyspark import SparkConf
from pyspark.sql import SparkSession

master = "local[*]"
app_name = "FIT5202-Week4-ParallelJoins"

spark_conf = SparkConf().setMaster(master).setAppName(app_name)

spark = SparkSession.builder.config(conf=spark_conf).getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("SparkSession created.")

### From filtering one DataFrame to matching two

Recall from Week 3 lecture:
- A filter can normally process each partition of a single DataFrame **independently**.
- A join needs matching rows from **two** DataFrames to be compared together. Spark must bring matching keys into the same place, either by **repartitioning both sides** or by **broadcasting** one of them to every executor.

Let's practice this with two small, hand-built tables.

In [ ]:
athletes_data = [
    (1, "Ariarne Titmus",  "AUS"),
    (2, "Katie Ledecky",   "USA"),
    (3, "Adam Peaty",      "GBR"),
    (4, "Emma McKeon",     "AUS"),
    (5, "Yui Ohashi",      "JPN"),
]

medals_data = [
    (101, 1,  "Gold"),
    (102, 1,  "Silver"),
    (103, 2,  "Gold"),
    (104, 3,  "Gold"),
    (105, 4,  "Gold"),
    (106, 4,  "Bronze"),
    (107, 99, "Bronze"),  # references an athlete_id that does not exist in athletes_df
]

athletes_df = spark.createDataFrame(athletes_data, ["athlete_id", "athlete_name", "country"])
medals_df = spark.createDataFrame(medals_data, ["medal_id", "athlete_id", "medal_type"])

athletes_df.show()
medals_df.show()

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px">
<strong style="color:#FF5555">Task: </strong> Looking at the two tables above, answer:

<ol>
<li>Which column is the join key between these two tables?</li>
<li>Which <code>athlete_id</code> value(s) repeat in <code>medals_df</code>?</li>
<li>Which row(s), in either table, have no match on the other side?</li>
</ol>

<strong>YOUR ANSWER HERE.</strong>
</div>

In [ ]:
# 1. Join key:


# 2. Repeated athlete_id value(s):


# 3. Row(s) with no match:

<a id="part-2"></a>
# Part 2 — Inner and Outer Joins (10 mins)

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"> <strong style="color:#006DAE">Predict, before running the next two cells:</strong> using your answers from Part 1, how many rows will an <b>inner</b> join of <b>athletes_df</b> and <b>medals_df</b> return? How many rows will a <b>left</b> join return? </div>

In [ ]:
# YOUR PREDICTION HERE
# Inner join row count:
# Left join row count:

### Inner join

An inner join keeps only rows where the join key is present on **both** sides.

In [ ]:
inner_df = athletes_df.join(
    medals_df,
    on="athlete_id",
    how="inner"
)
inner_df.orderBy("athlete_id").show()
print("Row count:", inner_df.count())

### Left outer join

A left outer join keeps **every** row from the left DataFrame (`athletes_df`), filling in `NULL` for any right-side columns that have no match.

In [ ]:
left_df = athletes_df.join(
    medals_df,
    on="athlete_id",
    how="left"
)
left_df.orderBy("athlete_id").show()
print("Row count:", left_df.count())

### Full outer join

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"> <strong style="color:#006DAE">Comparison question:</strong> compared with the left join above, what <b>additional</b> row would a full outer join preserve, and why does that row only appear in a full outer join? </div>

In [ ]:
# YOUR ANSWER HERE
# Compare

In [ ]:
# full outer join
full_df = athletes_df.join(
    medals_df,
    on="athlete_id",
    how="full"
)
full_df.orderBy("athlete_id").show()
print("Row count:", full_df.count())

>The inner and left joins differ in which unmatched rows they preserve. However, the join type does not determine the physical algorithm Spark uses to produce the result.

The next section examines this distinction between the logical join and its physical execution.

<a id="part-3"></a>
# Part 3 — Logical Join versus Physical Strategy (15 mins)

So far we have only asked *which rows* a join should return: that is the join's **logical semantics**. It says nothing about *how* Spark actually produces those rows.

| Level | Question |
|---|---|
| Logical join | Which rows must be returned? |
| Join key | Which values define a match? |
| Physical strategy | How will Spark perform the join? |
| Data movement | What does `Exchange` or `BroadcastExchange` indicate? |

An inner join is a **logical** requirement. Several different **physical strategies** can satisfy that same requirement:

| Physical strategy | Main data movement | Local matching method |
|---|---|---|
| `BroadcastHashJoin` | Broadcast the smaller relation to every executor | Build/probe a hash table |
| `SortMergeJoin` | Shuffle both sides by the join key | Sort and merge matching keys |
| `ShuffledHashJoin` | Shuffle both sides by the join key | Build a local hash table per partition |

Spark normally selects a supported physical strategy using planning rules, estimates, statistics and configuration. Developers can provide hints, and AQE may revise parts of the strategy using runtime evidence.

In [ ]:
inner_df.explain(mode="formatted")

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px">
<strong style="color:#FF5555">Task: </strong> Using the formatted plan above, identify:

<ol>
<li>the logical join type;</li>
<li>the join key;</li>
<li>the physical join operator Spark selected;</li>
<li>whether the plan contains an <code>Exchange</code> or a <code>BroadcastExchange</code>;</li>
<li>what this tells you about what data Spark intends to move, given that both <b>athletes_df</b> and <b>medals_df</b> are tiny, hand-built tables.</li>
</ol>

<strong>YOUR ANSWER HERE.</strong>
</div>

In [ ]:
# 1. Logical join type:
# 2. Join key:
# 3. Physical join operator:
# 4. Exchange or BroadcastExchange present:
# 5. What this tells you:

<a id="part-4"></a>
# Part 4 — Spark Join Planning and AQE (20 mins)

For the rest of the lab we will work with three real files describing over a century of Olympic medals:

- **`summer.csv`**: one row per medal awarded at the Summer Olympics (Year, City, Sport, Discipline, Athlete, Country, Gender, Event, Medal);
- **`winter.csv`**: the same structure, for the Winter Olympics;
- **`dictionary.csv`**: one row per country (Country name, three-letter Code, Population, GDP per Capita).

The join key connecting them is a three-letter National Olympic Committee (NOC) code: `summer.Country` and `winter.Country` hold this code, and `dictionary.Code` holds the same code for the country lookup table.

In [ ]:
dictionary_df = spark.read.csv("resources/dictionary.csv", header=True, inferSchema=True)
summer_df = spark.read.csv("resources/summer.csv", header=True, inferSchema=True).repartition(8)
winter_df = spark.read.csv("resources/winter.csv", header=True, inferSchema=True).repartition(4)

# summer.csv/winter.csv and dictionary.csv both use the column name "Country" for two different things: 
# a three-letter NOC code in the medal tables, and a full country name in the dictionary.
# Rename the dictionary's version now, before it causes an ambiguous-column error later. 
# Checking for overlapping column names before joining is good practice with any two real-world datasets.
dictionary_df = dictionary_df.withColumnRenamed("Country", "Country_Name")

summer_df = summer_df.cache()
winter_df = winter_df.cache()

sc.setJobDescription("Part 4 - Dataset setup and cache materialisation")
try:
    print("dictionary_df -> rows:", dictionary_df.count(), " partitions:", dictionary_df.rdd.getNumPartitions())
    print("summer_df     -> rows:", summer_df.count(),     " partitions:", summer_df.rdd.getNumPartitions())
    print("winter_df     -> rows:", winter_df.count(),     " partitions:", winter_df.rdd.getNumPartitions())
finally:
    sc.setJobDescription(None)

`dictionary_df` has 201 rows and comfortably fits on a single partition: a small **lookup table**. `summer_df` has 31,165 rows across 8 partitions. On disk, `dictionary.csv` is roughly 7.6 KB, while `summer.csv` is roughly 2.6 MB.

AQE remains **enabled** for the rest of this lab; it is Spark's default and we have no reason to turn it off for normal use. Two settings mainly determine what we will see:

In [ ]:
print("Adaptive execution enabled:", spark.conf.get("spark.sql.adaptive.enabled"))
print("Auto-broadcast threshold:", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px">
<strong style="color:#006DAE">Predict before running the next cell:</strong>

<ol>
<li>Which DataFrame is smaller: <code>summer_df</code> or <code>dictionary_df</code>?</li>
<li>Which physical join strategy do you expect Spark to select?</li>
<li>Which relation do you expect Spark to move?</li>
<li>Do you expect Spark to shuffle <code>summer_df</code> by <code>Country</code>?</li>
</ol>
</div>

In [ ]:
# YOUR PREDICTION HERE

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px">
<strong style="color:#FF5555">Task:</strong>

Create <code>auto_join_df</code> by performing an <strong>inner join</strong> between:

<ul>
<li><code>summer_df</code>, using <code>Country</code>; and</li>
<li><code>dictionary_df</code>, using <code>Code</code>.</li>
</ul>

Do not provide a broadcast hint or specify a physical join strategy. Spark should select the physical strategy using the available information.

After creating the DataFrame, display its formatted physical plan using <code>explain()</code>.
</div>

In [ ]:
# YOUR ANSWER HERE
auto_join_df = summer_df.join(
    # Add the second DataFrame
    

    # Add the join condition
   

    # Specify the logical join type
)

# Display the formatted physical plan
auto_join_df.explain(mode=____________________)

In [ ]:
sc.setJobDescription("Part 4 - Auto join (unhinted)")
try:
    row_count = auto_join_df.count()
finally:
    sc.setJobDescription(None)

print("Matched rows:", row_count)

Spark selected `BroadcastHashJoin` in the initial physical plan. AQE remains enabled and may revise supported parts of the plan using runtime statistics, although no revision is required when the initial plan is already suitable.

<a id="part-5"></a>
# Part 5 — Follow the Join in the Spark UI (20 mins)

Open the Spark UI at **http://localhost:4040** and find the job labelled `Part 4 - Auto join (unhinted)`. 

### SQL / DataFrame tab
Open the query for this job and find:
- the `BroadcastHashJoin` box;
- the `BroadcastExchange` box feeding into it;
- the output-row count entering the join from each side.

### Stages tab
Find the stage that reads `summer.csv` (or its cached copy) and check:
- how many tasks it contains, and whether that matches `summer_df`'s partition count;
- whether that stage shows a shuffle caused by the **join** itself, as opposed to `summer_df`'s original `repartition(8)`.

### Tasks table
Within that stage, compare:
- input records per task as the primary evidence of workload balance;
- task duration as supporting evidence, allowing for normal local execution noise.

> Exact job IDs, stage IDs and timing metrics may vary across machines and Spark versions. Use the job description and the physical operators you already identified in Part 4 to find the right job, rather than a fixed stage number.

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px">
<strong style="color:#FF5555">Task: </strong> Record what you found:

<ol>
<li>Output-row count leaving <code>BroadcastExchange</code>:</li>
<li>Number of tasks in the stage reading <code>summer_df</code>:</li>
<li>Does that stage show a shuffle caused by the join itself?</li>
<li>Did the tasks process reasonably similar numbers of input records? Are their durations broadly similar, allowing for normal local execution noise?</li>
</ol>

<strong>YOUR ANSWER HERE.</strong>
</div>

In [ ]:
# 1. BroadcastExchange output-row count:
# 2. Number of tasks:
# 3. Shuffle caused by the join?
# 4. Task durations even?

<a id="part-6"></a>
# Part 6 — Compare Broadcast Hash and Sort-Merge Joins (25 mins)

In Part 4, Spark automatically selected a broadcast hash join. We will now run the same logical join under conditions that produce a sort-merge join, then compare the two physical plans.

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px">
<strong style="color:#006DAE">About this experiment:</strong>

This experiment temporarily disables automatic broadcasting by setting the broadcast threshold to <code>-1</code>. AQE remains enabled, but broadcast conversion is unavailable while this setting is in effect.

The original settings are restored after the comparison. This is a controlled experiment for observing a different physical strategy, not a recommended configuration for normal use.
</div>

In [ ]:
# Save the original settings so we can restore them afterwards.
original_broadcast_threshold = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
original_shuffle_partitions = spark.conf.get("spark.sql.shuffle.partitions")

# Disable auto-broadcast, and use a small, fixed shuffle-partition count 
# so the resulting stage has a manageable number of tasks to inspect in the Spark UI.
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.conf.set("spark.sql.shuffle.partitions", 8)

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px">
<strong style="color:#FF5555">Task: Create the controlled sort-merge join</strong>

<p>Create <code>forced_shuffle_df</code> by performing the same inner join used in Part 4:</p>

<ul>
<li>join <code>summer_df</code> with <code>dictionary_df</code>;</li>
<li>match <code>summer_df.Country</code> with <code>dictionary_df.Code</code>;</li>
<li>use an inner join; and</li>
<li>display the formatted physical plan.</li>
</ul>

<p>Automatic broadcasting has been temporarily disabled. Inspect the plan to determine which physical join strategy Spark selects under this setting.</p>
</div>

In [ ]:
# Create forced_shuffle_df using the same logical join as Part 4
# YOUR CODE HERE


# Display the formatted physical plan
# YOUR CODE HERE

In [ ]:
sc.setJobDescription("Part 6 - Forced shuffle join")
try:
    forced_row_count = forced_shuffle_df.count()
finally:
    sc.setJobDescription(None)

print("Matched rows (forced shuffle):", forced_row_count)

In [ ]:
# Restore the original configuration immediately after the controlled experiment.
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", original_broadcast_threshold)
spark.conf.set("spark.sql.shuffle.partitions", original_shuffle_partitions)

print("Restored auto-broadcast threshold:", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))
print("Restored shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

### Compare the results

The automatic and controlled versions use:

- the same two DataFrames;
- the same join condition;
- the same inner join.

Only the physical execution strategy is different. The automatic version uses a broadcast hash join, while the controlled version uses a sort-merge join.

Compare the row counts printed for the two executions.

> The physical strategy changes how Spark performs the join, but it does not change the logical result requested by the query.

Matching row counts are sufficient for this controlled comparison because only the physical strategy was changed. When a query is rewritten or optimised manually, a stronger result comparison may also be required.

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px">
<strong style="color:#FF5555">Task: </strong> Complete the table below using the two physical plans.

<strong>YOUR ANSWER HERE.</strong>
</div>


| Evidence | Automatic broadcast join | Controlled sort-merge join |
|---|---|---|
| Physical join operator | | |
| `Exchange` operators | | |
| Which relation or relations move? | | |
| Is sorting required? | | |
| Main trade-off | | |

### Check the shuffle in the Spark UI

Open the job labelled `Part 6 - Forced shuffle join`.
You do not need to repeat the complete Spark UI walkthrough from Part 5. Check:

1. whether the sort-merge join introduces shuffle read and write;
2. how many tasks run in the stage following the shuffle;
3. how this differs from the automatic broadcast join.


### What if the join key is unevenly distributed (skew)?

A frequently occurring join key can make one shuffle partition much larger than the others.

> If `USA` occurs much more frequently than other country codes, under which strategy from Part 6 would its rows be more likely to concentrate in one join partition: broadcast hash join or sort-merge join? Explain why.

We will examine key frequencies in the next lab.

<a id="part-7"></a>
# Part 7 — Check an Optimisation Suggestion (10 mins)

You will increasingly encounter optimisation suggestions from colleagues, code review comments, or AI coding assistants. Some will be correct, some will be outdated, and some will be based on a misreading of what is actually happening. 
Consider the following suggestion:
<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px">
<strong style="color:#FF5555">Suggestion:</strong>

“The join uses <code>SortMergeJoin</code>, so adding <code>broadcast(dictionary_df)</code> will always make it faster.”
</div>

Your task is to audit it using the DataFrame sizes and physical plans from the previous sections to decide whether this suggestion is appropriate.

In [ ]:
from pyspark.sql.functions import broadcast

hinted_join_df = summer_df.join(
    broadcast(dictionary_df),
    summer_df.Country == dictionary_df.Code,
    how="inner"
)
hinted_join_df.explain(mode="formatted")

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px">
<strong style="color:#FF5555">Task: </strong> Answer the following, then complete the table below.

<ol>
<li>Is <code>dictionary_df</code> genuinely small? What evidence do you have?</li>
<li>Did Spark already broadcast it in the unhinted join (Part 4)?</li>
<li>Would the hint above change that plan?</li>
<li>Would this recommendation be safe if <code>dictionary_df</code> were replaced with a much larger table?</li>
<li>If a <code>SortMergeJoin</code> appears somewhere (as it did in Part 6), does that alone prove the plan is poor?</li>
</ol>

<strong>YOUR ANSWER HERE.</strong>
</div>

| Item | Your response |
|---|---|
| What Spark selected automatically | |
| Evidence from the plan | |
| Whether the hint changes the plan | |
| Main risk | |

**Based on the evidence, would you accept, reject or conditionally accept the suggestion? Explain your decision.**

In [ ]:
# YOUR ANSWERS TO QUESTIONS 1-5 AND FINAL DECISION HERE

<a id="take-home"></a>
# Take-Home Practice

Today's lab focused on how Spark plans, executes and adapts a join. The following exercises give you more practice with the same ideas.

### 1. Catalyst optimisation around a join

In Week 3, you compared queries in which `filter()` and `select()` were written in different orders. Here, you will make a similar comparison using a join.

- **Query A**  join the DataFrames, then filter for gold medals.
- **Query B** filter for gold medals, then join the DataFrames.

**Before running:** will Spark actually filter *after* the join for Query A, exactly as written, or can the optimiser move the filter earlier?

For both queries:

<ul>
<li>join <code>summer_df.Country</code> with <code>dictionary_df.Code</code>;</li>
<li>select <code>Athlete</code>, <code>Country</code> and <code>Medal</code>;</li>
<li>display the extended execution plan using <code>explain(mode="extended")</code>.</li>
</ul>
</div>

In [ ]:
from pyspark.sql.functions import col

# Query A: join first, then filter for gold medals
# YOUR CODE HERE



# Query B: filter for gold medals first, then join
# YOUR CODE HERE



# Display the extended plans for both queries
print("===== QUERY A: join, then filter =====")
# YOUR CODE HERE


print("\n\n===== QUERY B: filter, then join =====")
# YOUR CODE HERE

**Question:** 
Compare the **Parsed Logical Plans** and **Optimized Logical Plans**  for Query A and Query B. Did Catalyst move the filter in Query A? Are the two physical plans equivalent in their relevant operators and operation order?



**YOUR ANSWER HERE.**

In [ ]:
print("Query A row count:", query_a.count())
print("Query B row count:", query_b.count())

### 2. Left-semi and left-anti joins with a real business question

A **left-semi join** returns rows from the left DataFrame that **have** a match on the right, keeping only the left DataFrame's columns. A **left-anti join** returns rows from the left DataFrame that **do not** have a match on the right.

You have not used `winter_df` yet today. Use it to answer two business questions:

1. Which countries in our dictionary have won **at least one** Winter Olympics medal? (left-semi)
2. Which countries in our dictionary have **never** won a Winter Olympics medal? (left-anti)

In [ ]:
# 1. Countries WITH at least one Winter medal (left_semi join of dictionary_df and winter_df)
# YOUR CODE HERE


# 2. Countries with NO Winter medal on record (left_anti join)
# YOUR CODE HERE

### 3. Optional extension: manual repartitioning

`winter_df` was loaded with `.repartition(4)` earlier today.

**Question:** if you `repartition(200, "Country")` on `winter_df` immediately before joining it with `dictionary_df`, does `repartition()` **remove** a shuffle the join would otherwise need, or does it **add another one**? Use `explain(mode="formatted")` to check, and explain your reasoning.

**YOUR ANSWER HERE.**

In [ ]:
# YOUR CODE HERE

<a id="stop-spark"></a>
## Stop Spark

Run the next cell only when you have completely finished the notebook.

We first release the cached DataFrames explicitly, then stop the SparkSession, which releases the remaining resources.

In [ ]:
summer_df.unpersist()
winter_df.unpersist()

spark.stop()
print(
    "SparkSession stopped. Congratulations! Today you moved from filtering a single DataFrame to joining two! "
    "See you next week, when we build on this foundation with parallel aggregation and groupBy()."
)